In [ ]:
from pyuvdata import UVData
import numpy as np
import json
import xcorr_cpu_tb as xc
import importlib
import helper
from albatros_analysis.src.utils import orbcomm_utils as outils

In [2]:
working_directory = '/project/s/sievers/thomasb/mars_data_23'
bits = 1
baseline_idx = 1
day = '07_11'
config_file_name = f'config_{day}.json'

In [16]:
importlib.reload(xc)
importlib.reload(helper)
pols, rowcounts, channels, info = xc.run_script('config3.json')

ALB3 ALB1 {'name': 'Antenna 1', 'path': '/project/s/sievers/albatros/mars/202307/baseband/stn_1_central', 'coordinates': [10.123, -20.456, 30.789], 'clock_offset': 0}
ALB3 ALB2 {'name': 'Antenna 2', 'path': '/project/s/sievers/albatros/mars/202307/baseband/stn_2_east', 'coordinates': [11.234, -21.567, 31.89], 'clock_offset': -277647}
ALB3 ALB3 {'name': 'Antenna 3', 'path': '/project/s/sievers/albatros/mars/202307/baseband/stn_3_west', 'coordinates': [12.345, -22.678, 32.901], 'clock_offset': -603518}
took 0.187 seconds to read raw data on  /project/s/sievers/albatros/mars/202307/baseband/stn_1_central/16996/1699625038.raw
took 0.344 seconds to read raw data on  /project/s/sievers/albatros/mars/202307/baseband/stn_2_east/16996/1699625013.raw
took 0.391 seconds to read raw data on  /project/s/sievers/albatros/mars/202307/baseband/stn_3_west/16996/1699624999.raw
before correction 427246 1953125
-22604
after correction 427246 1975729
before correction 427246 2807617
3412
after correction 4

In [ ]:
#figure out if there are missing values for Nblts (is it always full?)
    #currently, flag array is just the given mask
#translate channels into frequency values
#

[4.88281250e+06 4.94384766e+06 5.00488281e+06 5.06591797e+06
 5.12695312e+06 5.18798828e+06 5.24902344e+06 5.31005859e+06
 5.37109375e+06 5.43212891e+06 5.49316406e+06 5.55419922e+06
 5.61523438e+06 5.67626953e+06 5.73730469e+06 5.79833984e+06
 5.85937500e+06 5.92041016e+06 5.98144531e+06 6.04248047e+06
 6.10351562e+06 6.16455078e+06 6.22558594e+06 6.28662109e+06
 6.34765625e+06 6.40869141e+06 6.46972656e+06 6.53076172e+06
 6.59179688e+06 6.65283203e+06 6.71386719e+06 6.77490234e+06
 6.83593750e+06 6.89697266e+06 6.95800781e+06 7.01904297e+06
 7.08007812e+06 7.14111328e+06 7.20214844e+06 7.26318359e+06
 7.32421875e+06 7.38525391e+06 7.44628906e+06 7.50732422e+06
 7.56835938e+06 7.62939453e+06 7.69042969e+06 7.75146484e+06
 7.81250000e+06 7.87353516e+06 7.93457031e+06 7.99560547e+06
 8.05664062e+06 8.11767578e+06 8.17871094e+06 8.23974609e+06
 8.30078125e+06 8.36181641e+06 8.42285156e+06 8.48388672e+06
 8.54492188e+06 8.60595703e+06 8.66699219e+06 8.72802734e+06
 8.78906250e+06 8.850097

In [ ]:
# Initialize UVData object
uv = UVData()
global_start_time = 0

acclen = info["acclen"]
nchunks = info["nchunks"]
t_acclen = info["t_acclen"]
chanstart = info["chanstart"]
chanend = info["chanend"]

uv.Ntimes, uv.Nbls, uv.Npols, uv.Nfreqs = pols.shape
uv.Nblts = uv.Ntimes * uv.Nbls




uv.Nants_data = 3  #somehow make automatic
uv.Nants_telescope = 3


uv.antenna_names = ['ant0', 'ant1', 'ant2']
uv.antenna_numbers = [0,1,2]
uv.antenna_positions = np.zeros((3,3))


nt = uv.Ntimes
vis_list = []
flag_list = []
ant1_list = []
ant2_list = []
code_list = []
time_list = []
nsample_list = []

idx = 0
for i in range(uv.Nants_data):
    for j in range(i+1,uv.Nants_data):
        data = pols.data[:,idx,:,:].transpose(0,2,1)
        flags = pols.mask[:,idx,:,:].transpose(0,2,1)
        nsamples = rowcounts[:,idx]
        vis_list.append(data)
        flag_list.append(flags)
        nsample_list.append(nsamples)

        ant1_list.extend([i] * nt)
        ant2_list.extend([j] * nt)

        code = int(i*2048 + j + 2**(16))
        code_list.extend([code] * nt)

        times = np.linspace(global_start_time, global_start_time + (uv.Ntimes-1)*t_acclen, uv.Ntimes)
        time_list.append(times)

        #ROWCOUNTS: I think there might be an error. there should be a rowcount thing for each baseline.

        print((i,j), idx)
        idx += 1


uv.data_array = np.concatenate(vis_list)
uv.flag_array = np.concatenate(flag_list)
uv.ant_1_array = np.array(ant1_list)
uv.ant_2_array = np.array(ant2_list)
uv.baseline_array = np.array(code_list)
uv.time_array = np.concatenate(time_list)
uv.lst_array = uv.time_array   #need to add LST time into this!!

nsample_array = np.concatenate(nsample_list)
uv.nsample_array = np.tile(nsample_array[:, np.newaxis, :], (1, uv.Nfreqs, 1)) # same nsample for each channel, different for each pol, baseline
uv.nsample_array = uv.nsample_array.astype(float)

uv.integration_time = np.full(uv.Nblts, t_acclen)

print(uv.data_array.shape)



#frequency stuff
assert uv.Nfreqs == uv.data_array.shape[1]  #safety check

width = 250e6 / 4096

freqs = np.zeros(uv.Nfreqs)
for i in range(uv.Nfreqs):
    freqs[i] = outils.chan2freq(channels[i])
uv.freq_array = freqs
uv.channel_width = np.full(uv.Nfreqs, width)


#ADD WHICH EXACT TYPE OF POLARIZATION WE ARE WORKING WITH
uv.polarization_array = np.arange(1,3, dtype = int)  # e.g., XX


#TBD
uv.Nspws = 1
uv.Nphase = 1
#uv.Nants_telescope = 2


#spectral window stuff
uv.spw_array = np.array([0])
uv.flex_spw_id_array = np.zeros((uv.Nfreqs), dtype=int)


#metadata
uv.telescope_name = 'FakeTelescope'
uv.instrument = 'FakeInstrument'
uv.telescope_location = (0.0, 0.0, 0.0)
uv.history = 'History'
uv.object_name = 'Object Name'
uv.vis_units = 'uncalib'



#bare minimum phase center content
uv.Nphase = 1
uv.phase_center_catalog = {
    0: {
        "cat_name": "zenith",        # Any name
        "cat_type": "sidereal",      # Required; options: 'sidereal', 'ephem', 'driftscan'
        "cat_lon": 0.0,              # RA
        "cat_lat": 0.0,              # Dec
        "cat_frame": "icrs",         # 'icrs' is safe and standard
    }
}
uv.phase_center_id_array = np.zeros(uv.Nblts, dtype=int)
uv.phase_center_app_ra = np.zeros(uv.Nblts)
uv.phase_center_app_dec = np.zeros(uv.Nblts)
uv.phase_center_frame_pa = np.zeros(uv.Nblts)
uv.uvw_array = np.ones((uv.Nblts, 3), dtype = float)


# Write to UVH5 file
uv.write_uvh5('example_output.uvh5', clobber=True)

(0, 1) 0
(0, 2) 1
(1, 2) 2
(639, 288, 2)
File exists; clobbering


itrs position vector magnitudes must be on the order of the radius of Earth -- they appear to lie well below this.
ERFA function "utcut1" yielded 213 of "dubious year (Note 3)"
ERFA function "utctai" yielded 213 of "dubious year (Note 3)"
The lst_array is not self-consistent with the time_array and telescope location. Consider recomputing with the `set_lsts_from_time_array` method.
